In [2]:
###신용카드 이상감지 모델을 만들기 위함.
##1. 사기거래 여부를 예측하는 데 불필요한 컬럼을 제거합니다.
##2. 최소 2개 이상의 새로// 운 피처(변수)를 생성해봅시다. [힌트] 구매 금액, 시간 등의 변수를 이용할 수 있습니다.
##3. 데이터의 위도/경도 정보를 활용해볼 수 있을까요? (인터넷 검색 적극 활용)
##4. 통계적 관점으로 접근하여 유용한 변수를 만들어낼 수 있을까요?

In [3]:
import sys
import pandas as pd
import numpy as np 
import seaborn as sns
import matplotlib.pyplot as plt

In [4]:
df = pd.read_csv('fraud.csv')

In [5]:
df = df.rename(columns={
    'trans_date_trans_time': '거래일시',
    'trans_num': '거래ID',
    'amt': '거래금액',
    'unix_time': '유닉스타임스탬프',
    
    'cc_num': '카드번호',
    'first': '이름',
    'last': '성',
    'gender': '성별',
    'dob': '생년월일',
    'job': '직업',
    
    'street': '주소',
    'city': '도시',
    'state': '주',
    'zip': '우편번호',
    'lat': '위도',
    'long': '경도',
    'city_pop': '도시인구',
    
    'merchant': '가맹점명',
    'category': '결제_단말기',
    'merch_lat': '가맹점_위도',
    'merch_long': '가맹점_경도',
    
    'is_fraud': '사기여부'
})

In [6]:
pd.set_option('display.max_columns', None)

print(df.shape)
df.head()

(491134, 22)


,거래일시,카드번호,가맹점명,결제_단말기,거래금액,이름,성,성별,주소,도시,주,우편번호,위도,경도,도시인구,직업,생년월일,거래ID,유닉스타임스탬프,가맹점_위도,가맹점_경도,사기여부
0,2019-01-01 00:00:44,630423337322,"fraud_Heller, Gutmann and Zieme",grocery_pos,107.23,Stephanie,Gill,F,43039 Riley Greens Suite 393,Orient,WA,99160,48.8878,-118.2105,149,Special educational needs teacher,1978-06-21,1f76529f8574734946361c461b024d99,1325376044,49.159047,-118.186462,0
1,2019-01-01 00:12:34,4956828990005111019,"fraud_Schultz, Simonis and Little",grocery_pos,44.71,Kenneth,Robinson,M,269 Sanchez Rapids,Elizabeth,NJ,7208,40.6747,-74.2239,124967,Operational researcher,1980-12-21,09eff9c806365e2a6be12c1bbab3d70e,1325376754,40.079588,-74.848087,0
2,2019-01-01 00:17:16,180048185037117,fraud_Kling-Grant,grocery_net,46.28,Mary,Wall,F,2481 Mills Lock,Plainfield,NJ,7060,40.6152,-74.4150,71485,Leisure centre manager,1974-07-19,19e23c6a300c774354417befe4f31f8c,1325377036,40.021888,-74.228188,0
3,2019-01-01 00:20:15,374930071163758,fraud_Deckow-O'Conner,grocery_pos,64.09,Daniel,Escobar,M,61390 Hayes Port,Romulus,MI,48174,42.2203,-83.3583,31515,Police officer,1971-11-05,6f363661ba6b55889e488dd178f2a0af,1325377215,42.360426,-83.552316,0
4,2019-01-01 00:23:41,2712209726293386,fraud_Balistreri-Nader,misc_pos,25.58,Jenna,Brooks,F,50872 Alex Plain Suite 088,Baton Rouge,LA,70808,30.4066,-91.1468,378909,"Designer, furniture",1977-02-22,1654da2abfb9e79a5f99167fc9779558,1325377421,29.737426,-90.853194,0


In [7]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 491134 entries, 0 to 491133
Data columns (total 22 columns):
 #   Column    Non-Null Count   Dtype  
---  ------    --------------   -----  
 0   거래일시      491134 non-null  object 
 1   카드번호      491134 non-null  int64  
 2   가맹점명      491134 non-null  object 
 3   결제_단말기    491134 non-null  object 
 4   거래금액      491134 non-null  float64
 5   이름        491134 non-null  object 
 6   성         491134 non-null  object 
 7   성별        491134 non-null  object 
 8   주소        491134 non-null  object 
 9   도시        491134 non-null  object 
 10  주         491134 non-null  object 
 11  우편번호      491134 non-null  int64  
 12  위도        491134 non-null  float64
 13  경도        491134 non-null  float64
 14  도시인구      491134 non-null  int64  
 15  직업        491134 non-null  object 
 16  생년월일      491134 non-null  object 
 17  거래ID      491134 non-null  object 
 18  유닉스타임스탬프  491134 non-null  int64  
 19  가맹점_위도    491134 non-null  float64
 20  가맹점_

In [8]:
df.describe()

,카드번호,거래금액,우편번호,위도,경도,도시인구,유닉스타임스탬프,가맹점_위도,가맹점_경도,사기여부
count,4.911340e+05,491134.000000,491134.000000,491134.000000,491134.000000,4.911340e+05,4.911340e+05,491134.000000,491134.000000,491134.000000
mean,3.706013e+17,69.050120,50770.532384,37.931230,-90.495619,1.213922e+05,1.358730e+09,37.930272,-90.495411,0.002533
std,1.260229e+18,160.322867,26854.947965,5.341193,12.990732,3.725751e+05,1.819402e+07,5.372986,13.004100,0.050264
min,5.038744e+11,1.000000,1843.000000,24.655700,-122.345600,4.600000e+01,1.325376e+09,23.655789,-123.345106,0.000000
25%,2.131124e+14,8.960000,28405.000000,33.746700,-97.235100,1.228000e+03,1.343087e+09,33.781388,-96.984814,0.000000
50%,3.531130e+15,42.170000,49628.000000,38.507200,-87.591700,5.760000e+03,1.357257e+09,38.545124,-87.573441,0.000000
75%,4.653879e+15,80.330000,75048.000000,41.520500,-80.731000,5.083500e+04,1.374626e+09,41.624294,-80.685567,0.000000
max,4.956829e+18,25086.940000,99323.000000,48.887800,-69.965600,2.906700e+06,1.388534e+09,49.887523,-68.965624,1.000000


In [9]:
#head, info, describe 확인
#1. 카드번호, 가맹점명, 이름, 성, 주소, 우편번호, 거래ID, 유닉스타임스탬프처럼 랜덤부여되는 정보나 개인정보는 삭제. 
#   도시, 주는 사기비율과 연관지어 설명할 수 있을 수 있으므로 보류
#2. 결측치는 없고, Datetime 형식으로 변환이 필요한 변수는 거래일시, 생년월일이고 생년월일은 사기당시 나이로 변환.
#3. 사기 발생률이 0.002533 으로 사기비율이 매우 낮음.
#4. 거래금액은 달러로 되어있으므로 한화로 변환한 변수를 추가 생성.

df = df.drop(columns =  ['카드번호', '가맹점명', '이름', '성', '주소', '우편번호', '거래ID', '유닉스타임스탬프'])

df.head() #삭제확인

,거래일시,결제_단말기,거래금액,성별,도시,주,위도,경도,도시인구,직업,생년월일,가맹점_위도,가맹점_경도,사기여부
0,2019-01-01 00:00:44,grocery_pos,107.23,F,Orient,WA,48.8878,-118.2105,149,Special educational needs teacher,1978-06-21,49.159047,-118.186462,0
1,2019-01-01 00:12:34,grocery_pos,44.71,M,Elizabeth,NJ,40.6747,-74.2239,124967,Operational researcher,1980-12-21,40.079588,-74.848087,0
2,2019-01-01 00:17:16,grocery_net,46.28,F,Plainfield,NJ,40.6152,-74.4150,71485,Leisure centre manager,1974-07-19,40.021888,-74.228188,0
3,2019-01-01 00:20:15,grocery_pos,64.09,M,Romulus,MI,42.2203,-83.3583,31515,Police officer,1971-11-05,42.360426,-83.552316,0
4,2019-01-01 00:23:41,misc_pos,25.58,F,Baton Rouge,LA,30.4066,-91.1468,378909,"Designer, furniture",1977-02-22,29.737426,-90.853194,0


In [10]:
#Datetime 형식으로 생년월일과 거래일시를 변환함
df['거래일시'] = pd.to_datetime(df['거래일시'])
df['생년월일'] = pd.to_datetime(df['생년월일'])

#거래당시 나이 속성 추가 및 생년월일 삭제.
df['나이'] = (df['거래일시'] - df['생년월일']).dt.days // 365
df = df.drop(columns = ['생년월일'])

df.head() #삭제확인

,거래일시,결제_단말기,거래금액,성별,도시,주,위도,경도,도시인구,직업,가맹점_위도,가맹점_경도,사기여부,나이
0,2019-01-01 00:00:44,grocery_pos,107.23,F,Orient,WA,48.8878,-118.2105,149,Special educational needs teacher,49.159047,-118.186462,0,40
1,2019-01-01 00:12:34,grocery_pos,44.71,M,Elizabeth,NJ,40.6747,-74.2239,124967,Operational researcher,40.079588,-74.848087,0,38
2,2019-01-01 00:17:16,grocery_net,46.28,F,Plainfield,NJ,40.6152,-74.4150,71485,Leisure centre manager,40.021888,-74.228188,0,44
3,2019-01-01 00:20:15,grocery_pos,64.09,M,Romulus,MI,42.2203,-83.3583,31515,Police officer,42.360426,-83.552316,0,47
4,2019-01-01 00:23:41,misc_pos,25.58,F,Baton Rouge,LA,30.4066,-91.1468,378909,"Designer, furniture",29.737426,-90.853194,0,41


In [11]:
#거래일시를 확인해보니 2019-01-01 부터 2020-12-31 까지 2년치 데이터임
#순서대로 되어있는듯 보이지만 다시 정렬해서 확인처리해보는것이 좋다고 판단
df.head(10)
df.tail(10)
df = df.sort_values('거래일시') 
df

,거래일시,결제_단말기,거래금액,성별,도시,주,위도,경도,도시인구,직업,가맹점_위도,가맹점_경도,사기여부,나이
0,2019-01-01 00:00:44,grocery_pos,107.23,F,Orient,WA,48.8878,-118.2105,149,Special educational needs teacher,49.159047,-118.186462,0,40
1,2019-01-01 00:12:34,grocery_pos,44.71,M,Elizabeth,NJ,40.6747,-74.2239,124967,Operational researcher,40.079588,-74.848087,0,38
2,2019-01-01 00:17:16,grocery_net,46.28,F,Plainfield,NJ,40.6152,-74.4150,71485,Leisure centre manager,40.021888,-74.228188,0,44
3,2019-01-01 00:20:15,grocery_pos,64.09,M,Romulus,MI,42.2203,-83.3583,31515,Police officer,42.360426,-83.552316,0,47
4,2019-01-01 00:23:41,misc_pos,25.58,F,Baton Rouge,LA,30.4066,-91.1468,378909,"Designer, furniture",29.737426,-90.853194,0,41
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
491129,2020-12-31 23:56:48,home,134.26,F,Wilmington,NC,34.2651,-77.8670,186140,English as a second language teacher,34.853497,-78.664158,0,37
491130,2020-12-31 23:56:57,shopping_pos,25.49,F,Bradley,SC,34.0326,-82.2027,1523,Research scientist (physical sciences),35.008839,-81.475156,0,36
491131,2020-12-31 23:59:09,kids_pets,111.84,M,Lake Jackson,TX,29.0393,-95.4401,28739,Futures trader,29.661049,-96.186633,0,21
491132,2020-12-31 23:59:15,kids_pets,86.88,F,Burbank,WA,46.1966,-118.9017,3684,Musician,46.658340,-119.715054,0,39


In [12]:
df[df['사기여부'] == 1] #사기거래 피해자들의 데이터를 보니, 도시와 주가 비슷하고 나이대도 비슷한것을 볼 수 있음.

,거래일시,결제_단말기,거래금액,성별,도시,주,위도,경도,도시인구,직업,가맹점_위도,가맹점_경도,사기여부,나이
4794,2019-01-12 00:59:01,gas_transport,11.73,M,Cochranton,PA,41.5205,-80.0573,5507,Retail merchandiser,41.947427,-79.796264,1,45
4816,2019-01-12 03:48:07,grocery_pos,328.68,M,Cochranton,PA,41.5205,-80.0573,5507,Retail merchandiser,42.148618,-79.398595,1,45
4979,2019-01-12 15:46:10,food_dining,120.58,M,Cochranton,PA,41.5205,-80.0573,5507,Retail merchandiser,42.470024,-80.126576,1,45
5073,2019-01-12 19:53:59,shopping_net,1081.35,M,Cochranton,PA,41.5205,-80.0573,5507,Retail merchandiser,42.455406,-79.521640,1,45
5124,2019-01-12 22:44:05,shopping_net,776.70,M,Cochranton,PA,41.5205,-80.0573,5507,Retail merchandiser,40.680209,-79.099101,1,45
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
477832,2020-12-21 02:21:41,grocery_pos,358.24,F,Heart Butte,MT,48.2777,-112.8456,743,Water engineer,47.526202,-113.643313,1,48
477847,2020-12-21 02:36:03,shopping_net,859.12,F,Heart Butte,MT,48.2777,-112.8456,743,Water engineer,48.272348,-112.328075,1,48
479296,2020-12-21 22:38:38,home,209.84,F,Heart Butte,MT,48.2777,-112.8456,743,Water engineer,49.173669,-112.698767,1,48
479305,2020-12-21 22:42:11,food_dining,123.58,F,Heart Butte,MT,48.2777,-112.8456,743,Water engineer,48.913048,-113.214921,1,48


In [13]:
rate = 1200  # 1달러 = 1200원

df['거래금액(한화)'] = df['거래금액'] * rate

df.head() #거래금액(한화) 확인

,거래일시,결제_단말기,거래금액,성별,도시,주,위도,경도,도시인구,직업,가맹점_위도,가맹점_경도,사기여부,나이,거래금액(한화)
0,2019-01-01 00:00:44,grocery_pos,107.23,F,Orient,WA,48.8878,-118.2105,149,Special educational needs teacher,49.159047,-118.186462,0,40,128676.0
1,2019-01-01 00:12:34,grocery_pos,44.71,M,Elizabeth,NJ,40.6747,-74.2239,124967,Operational researcher,40.079588,-74.848087,0,38,53652.0
2,2019-01-01 00:17:16,grocery_net,46.28,F,Plainfield,NJ,40.6152,-74.4150,71485,Leisure centre manager,40.021888,-74.228188,0,44,55536.0
3,2019-01-01 00:20:15,grocery_pos,64.09,M,Romulus,MI,42.2203,-83.3583,31515,Police officer,42.360426,-83.552316,0,47,76908.0
4,2019-01-01 00:23:41,misc_pos,25.58,F,Baton Rouge,LA,30.4066,-91.1468,378909,"Designer, furniture",29.737426,-90.853194,0,41,30696.0


In [16]:
산점표 / 막대그래프 / 히스토그램 / 상관관계분석 / 시계열분석 등 다양한 방법으로 분석가능.
#분석 목적에 따른 접근방법 

활용 => 어떤것을 중점적으로 볼지를 정리하고 그에 맞춰서 분석해야함.



SyntaxError: invalid syntax (3489873727.py, line 1)

In [ ]:
# 하버사인 공식을 활용해 위도/경도 정보를 이용해 고객과 가맹점의 거리를 계산.
# 위도/경도는 도 단위이므로 라디안 단위로 변환 필요. drg2red = 라디안

def to_rad(x):
    return np.deg2rad(x)

R = 6371  # 지구 반지름 (km)

lat1 = to_rad(df['위도'])
lon1 = to_rad(df['경도'])
lat2 = to_rad(df['가맹점_위도'])
lon2 = to_rad(df['가맹점_경도'])

dlat = lat2 - lat1
dlon = lon2 - lon1

a = np.sin(dlat/2)**2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon/2)**2
c = 2 * np.arcsin(np.sqrt(a))

df['고객_가맹점_거리_km'] = R * c

df = df.drop(columns=['위도', '경도', '가맹점_위도', '가맹점_경도'])

df.head()

,거래일시,결제_단말기,거래금액,성별,도시,주,도시인구,직업,사기여부,나이,거래금액(한화),고객_가맹점_거리_km
0,2019-01-01 00:00:44,grocery_pos,107.23,F,Orient,WA,149,Special educational needs teacher,0,40,128676.0,30.212176
1,2019-01-01 00:12:34,grocery_pos,44.71,M,Elizabeth,NJ,124967,Operational researcher,0,38,53652.0,84.702120
2,2019-01-01 00:17:16,grocery_net,46.28,F,Plainfield,NJ,71485,Leisure centre manager,0,44,55536.0,67.847742
3,2019-01-01 00:20:15,grocery_pos,64.09,M,Romulus,MI,31515,Police officer,0,47,76908.0,22.303906
4,2019-01-01 00:23:41,misc_pos,25.58,F,Baton Rouge,LA,378909,"Designer, furniture",0,41,30696.0,79.591943


In [ ]:
#결제 단말기가 home이나, shopping_net등은 집에서 거래했을 수 있다고 판단되어 확인해본 결과 거리는 멀리 떨어져있음.

print(df['결제_단말기'].unique())
df[df['결제_단말기'] == 'entertainment']

['grocery_pos' 'grocery_net' 'misc_pos' 'gas_transport' 'misc_net'
 'health_fitness' 'travel' 'personal_care' 'shopping_net' 'shopping_pos'
 'home' 'entertainment' 'food_dining' 'kids_pets']


,거래일시,결제_단말기,거래금액,성별,도시,주,도시인구,직업,사기여부,나이,거래금액(한화),고객_가맹점_거리_km
174,2019-01-01 12:26:42,entertainment,6.26,F,Smiths Grove,KY,6841,"Therapist, sports",0,19,7512.0,85.130506
192,2019-01-01 12:50:07,entertainment,35.28,M,Hancock,MD,3766,Press photographer,0,34,42336.0,111.031405
205,2019-01-01 13:25:56,entertainment,38.52,F,Armagh,PA,922,Early years teacher,0,46,46224.0,87.408811
231,2019-01-01 14:01:04,entertainment,98.12,F,Roma,TX,18128,IT trainer,0,28,117744.0,116.690124
233,2019-01-01 14:03:23,entertainment,75.65,M,Thida,AR,111,Careers information officer,0,18,90780.0,126.405592
...,...,...,...,...,...,...,...,...,...,...,...,...
491098,2020-12-31 23:16:23,entertainment,24.72,F,Centerview,MO,2368,Electronics engineer,0,31,29664.0,2.825268
491102,2020-12-31 23:19:03,entertainment,92.34,M,Tekoa,WA,895,Clothing/textile technologist,0,21,110808.0,63.604186
491104,2020-12-31 23:22:18,entertainment,86.10,F,Armagh,PA,922,Early years teacher,0,48,103320.0,4.325912
491107,2020-12-31 23:25:48,entertainment,11.24,M,Espanola,NM,18408,Historic buildings inspector/conservation officer,0,48,13488.0,83.417302


In [ ]:
#상관관계 분석 // 유사한 데이터는 없음.

cols = ['거래금액', '거래금액(한화)', '나이', '도시인구', '고객_가맹점_거리_km']
df[cols].corr()

,거래금액,거래금액(한화),나이,도시인구,고객_가맹점_거리_km
거래금액,1.000000,1.000000,0.022658,0.013511,-0.001286
거래금액(한화),1.000000,1.000000,0.022658,0.013511,-0.001286
나이,0.022658,0.022658,1.000000,0.032539,0.003385
도시인구,0.013511,0.013511,0.032539,1.000000,0.018109
고객_가맹점_거리_km,-0.001286,-0.001286,0.003385,0.018109,1.000000


In [20]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 491134 entries, 0 to 491133
Data columns (total 15 columns):
 #   Column    Non-Null Count   Dtype         
---  ------    --------------   -----         
 0   거래일시      491134 non-null  datetime64[ns]
 1   결제_단말기    491134 non-null  object        
 2   거래금액      491134 non-null  float64       
 3   성별        491134 non-null  object        
 4   도시        491134 non-null  object        
 5   주         491134 non-null  object        
 6   위도        491134 non-null  float64       
 7   경도        491134 non-null  float64       
 8   도시인구      491134 non-null  int64         
 9   직업        491134 non-null  object        
 10  가맹점_위도    491134 non-null  float64       
 11  가맹점_경도    491134 non-null  float64       
 12  사기여부      491134 non-null  int64         
 13  나이        491134 non-null  int64         
 14  거래금액(한화)  491134 non-null  float64       
dtypes: datetime64[ns](1), float64(6), int64(3), object(5)
memory usage: 60.0+ MB
